In [21]:
!pip install optuna-integration[sklearn]

import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score,ParameterGrid,KFold
from sklearn.linear_model import  LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler ,MinMaxScaler
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import acf
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from optuna.integration import OptunaSearchCV
from optuna.distributions import IntDistribution, FloatDistribution, CategoricalDistribution


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [22]:
df=pd.read_csv('New Dataset/spain.csv')
df.head()


,Y,X,data_payload_id,instance_datetime,url,agency,platform_type,platform_id,platform_name,gaw_id,...,daily_utc_begin,daily_utc_end,daily_utc_mean,daily_nobs,daily_mmu,daily_columnso2,latest_observation,country,scientific_authority,version
0,28.46,-16.25,2306481,2025/03/05 00:00:00+00,https://woudc.org/archive/Archive-NewFormat/To...,AEMET,STN,401,Santa Cruz (Tenerife),SCO,...,9.15,10.52,9.60,4.0,2.181,-0.5,0,ESP,MIGUEL HERNANDEZ MARTINEZ DE LA PEÑA,0.0
1,28.46,-16.25,2306502,2025/03/05 00:00:00+00,https://woudc.org/archive/Archive-NewFormat/To...,AEMET,STN,401,Santa Cruz (Tenerife),SCO,...,12.50,17.88,15.22,12.0,1.421,-1.4,0,ESP,MIGUEL HERNANDEZ MARTINEZ DE LA PEÑA,0.0
2,28.46,-16.25,2306482,2025/03/05 00:00:00+00,https://woudc.org/archive/Archive-NewFormat/To...,AEMET,STN,401,Santa Cruz (Tenerife),SCO,...,8.85,14.03,10.80,16.0,1.638,-0.9,0,ESP,MIGUEL HERNANDEZ MARTINEZ DE LA PEÑA,0.0
3,28.46,-16.25,2306480,2025/03/05 00:00:00+00,https://woudc.org/archive/Archive-NewFormat/To...,AEMET,STN,401,Santa Cruz (Tenerife),SCO,...,12.83,17.27,15.41,10.0,1.602,-0.6,0,ESP,MIGUEL HERNANDEZ MARTINEZ DE LA PEÑA,0.0
4,28.46,-16.25,2306483,2025/03/05 00:00:00+00,https://woudc.org/archive/Archive-NewFormat/To...,AEMET,STN,401,Santa Cruz (Tenerife),SCO,...,9.55,17.55,15.04,19.0,1.680,-0.8,0,ESP,MIGUEL HERNANDEZ MARTINEZ DE LA PEÑA,0.0


In [23]:
df=df.drop(['Y', 'X', 'data_payload_id', 'instance_datetime', 'url', 'agency',
       'platform_type', 'platform_id', 'platform_name', 'gaw_id',
       'instrument_name', 'instrument_model', 'instrument_number',
       'monthly_date', 'monthly_stddevo3', 'monthly_npts','daily_stddevo3',
        'daily_wlcode', 'daily_obscode', 'monthly_columno3',
        'daily_utc_begin', 'daily_utc_end', 'daily_utc_mean',
       'daily_nobs', 'daily_mmu', 'daily_columnso2', 'latest_observation',
       'country', 'scientific_authority', 'version'], axis = 1)
df.head()

,daily_date,daily_columno3
0,2025-03-06,280.5
1,2025-03-27,310.2
2,2025-03-07,306.3
3,2025-03-05,319.4
4,2025-03-08,313.9


In [24]:
df=df.drop_duplicates()
df=df.dropna()
df.head()


,daily_date,daily_columno3
0,2025-03-06,280.5
1,2025-03-27,310.2
2,2025-03-07,306.3
3,2025-03-05,319.4
4,2025-03-08,313.9


In [25]:
print(f"Date Range: {df.loc[:,'daily_date'][len(df)-1]} to {df.loc[:,'daily_date'][0]}")

Date Range: 2001-03-10 to 2025-03-06


**# Trial 1** <br>
***TrainTestSplit + lag_70 + rolling_avg_7 + exp_avg_7*** <br>


In [26]:
# 2. Sort the dataframe by the full date (oldest to latest)
df1=df.copy()
df1 = df1.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df1['rolling_avg'] = df1['daily_columno3'].shift(1).rolling(window=7).mean()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day
df1['ema_avg'] = df1['daily_columno3'].shift(1).ewm(span=7, adjust=False).mean()


for i in range(1, 71):
    df1[f'lag{i}'] = df1['daily_columno3'].shift(i)


df1.dropna(inplace=True)
print(df1.shape)
df1.head()

(65440, 74)


,daily_date,daily_columno3,rolling_avg,ema_avg,lag1,lag2,lag3,lag4,lag5,lag6,...,lag61,lag62,lag63,lag64,lag65,lag66,lag67,lag68,lag69,lag70
70,1980-04-11,344.0,350.857143,345.432032,329.0,354.0,333.0,381.0,363.0,373.0,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
71,1980-04-14,360.0,353.857143,345.074024,344.0,329.0,354.0,333.0,381.0,363.0,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
72,1980-04-15,366.0,352.000000,348.805518,360.0,344.0,329.0,354.0,333.0,381.0,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
73,1980-04-16,398.0,352.428571,353.104139,366.0,360.0,344.0,329.0,354.0,333.0,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
74,1980-04-17,393.0,354.857143,364.328104,398.0,366.0,360.0,344.0,329.0,354.0,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [27]:
df1.tail()

,daily_date,daily_columno3,rolling_avg,ema_avg,lag1,lag2,lag3,lag4,lag5,lag6,...,lag61,lag62,lag63,lag64,lag65,lag66,lag67,lag68,lag69,lag70
65505,2025-03-31,304.2,318.200000,321.528513,337.5,323.5,296.8,314.8,317.0,310.6,...,401.5,407.4,312.9,398.9,386.8,327.4,358.0,411.6,351.7,368.0
65506,2025-03-31,326.3,314.914286,317.196385,304.2,337.5,323.5,296.8,314.8,317.0,...,309.3,401.5,407.4,312.9,398.9,386.8,327.4,358.0,411.6,351.7
65507,2025-03-31,332.6,317.157143,319.472289,326.3,304.2,337.5,323.5,296.8,314.8,...,403.9,309.3,401.5,407.4,312.9,398.9,386.8,327.4,358.0,411.6
65508,2025-03-31,291.9,319.385714,322.754217,332.6,326.3,304.2,337.5,323.5,296.8,...,300.6,403.9,309.3,401.5,407.4,312.9,398.9,386.8,327.4,358.0
65509,2025-03-31,325.0,316.114286,315.040662,291.9,332.6,326.3,304.2,337.5,323.5,...,411.2,300.6,403.9,309.3,401.5,407.4,312.9,398.9,386.8,327.4


In [28]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 71)]
lag_features.append('rolling_avg')
lag_features.append('ema_avg')
X = df1[lag_features]
y = df1['daily_columno3']


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
X_train = minmax_x.fit_transform(X_train)
y_train = minmax_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
X_test = minmax_x.transform(X_test)
y_test = minmax_y.transform(y_test.values.reshape(-1, 1)).flatten()

# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results[name] = {
        'Model': model,
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002660 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 18360
[LightGBM] [Info] Number of data points in the train set: 52352, number of used features: 72
[LightGBM] [Info] Start training from score 0.000000
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.649
MAE : 0.472
R2  : 52.69 %

----- Decision Tree -----
RMSE: 0.911
MAE : 0.664
R2  : 6.60 %

----- Random Forest -----
RMSE: 0.648
MAE : 0.473
R2  : 52.79 %

----- Gradient Boosting -----
RMSE: 0.648
MAE : 0.472
R2  : 52.83 %

----- Support Vector Regressor -----
RMSE: 0.669
MAE : 0.483
R2  : 49.64 %

----- XGBoost -----
RMSE: 0.662
MAE : 0.479
R2  : 50.76 %

----- LightGBM -----
RMSE: 0.649
MAE : 0.472
R2  : 52.56 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


**Trial 1: Tuning**

In [29]:

def tune_model(name, model_class, param_space):
    def objective(trial):
        params = {key: suggest(trial, key, dist) for key, dist in param_space.items()}
        model = model_class(**params)

        scores = []
        for train_idx, val_idx in cv.split(X_train):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            rmse = mean_squared_error(y_val, preds)
            scores.append(rmse)
        return np.mean(scores)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30, show_progress_bar=True)
    print(f"{name} Best Params: {study.best_params}")
    return model_class(**study.best_params)

def suggest(trial, name, dist):
    if isinstance(dist, tuple) and dist[0] == "int":
        return trial.suggest_int(name, *dist[1:])
    elif isinstance(dist, tuple) and dist[0] == "float":
        return trial.suggest_float(name, *dist[1:])
    elif isinstance(dist, tuple) and dist[0] == "cat":
        return trial.suggest_categorical(name, dist[1])
    else:
        raise ValueError(f"Unknown dist type for {name}: {dist}")

# Define parameter search spaces
param_spaces = {
    "XGBoost": {
        "tree_method": ("cat", ["gpu_hist"]),
        "predictor": ("cat", ["gpu_predictor"]),
        "n_estimators": ("int", 100, 150),
        "max_depth": ("int", 3, 7),
        "learning_rate": ("float", 0.01, 0.05),
        "subsample": ("float", 0.6, 0.9),
        "colsample_bytree": ("float", 0.6, 0.9)
    },
    "LightGBM": {  # Remove GPU
        "n_estimators": ("int", 100, 150),
        "learning_rate": ("float", 0.01, 0.05),
        "max_depth": ("int", 3, 7),
        "num_leaves": ("int", 15, 31),
        "subsample": ("float", 0.6, 0.9)
    },
    "Gradient Boosting": {
        "n_estimators": ("int", 100, 150),
        "learning_rate": ("float", 0.01, 0.1),
        "max_depth": ("int", 3, 5)
    },
    'Decision Tree': {
        'max_depth': ('int', 3, 10),
        'min_samples_split': ('int', 2, 10),
        'min_samples_leaf': ('int', 1, 5)
    },
    'Random Forest': {
        'n_estimators': ('int', 100, 200),
        'max_depth': ('int', 3, 10),
        'min_samples_split': ('int', 2, 10),
        'min_samples_leaf': ('int', 1, 5)
    },
    'Support Vector Regressor': {
        'kernel': ('cat', ['rbf']),
        'C': ('float', 0.1, 10.0),
        'epsilon': ('float', 0.01, 0.1)
    }
}

model_classes = {
    "XGBoost": XGBRegressor,
    "LightGBM": LGBMRegressor,
    "Gradient Boosting": GradientBoostingRegressor,
    'Decision Tree': DecisionTreeRegressor,
    'Random Forest': RandomForestRegressor,
    'Support Vector Regressor': SVR,
}

best_models = {}
results = {}

for name in model_classes:
    print(f"\nTuning {name}...")
    model = tune_model(name, model_classes[name], param_spaces[name])
    model.fit(X_train, y_train)
    best_models[name] = model

    # Evaluation
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# Print results
print("\nModel Evaluation Metrics:\n")
for name, metrics in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {metrics['RMSE']:.3f}")
    print(f"MAE : {metrics['MAE']:.3f}")
    print(f"R2  : {metrics['R2']*100:.2f} %\n")

[I 2025-06-03 01:41:03,006] A new study created in memory with name: no-name-15e6f720-b2e9-4fb3-a740-d42b6211edb2



Tuning XGBoost...


  0%|          | 0/30 [00:00<?, ?it/s]

Best trial: 0. Best value: 0.510493:   3%|▎         | 1/30 [00:01<00:36,  1.27s/it]

[I 2025-06-03 01:41:04,279] Trial 0 finished with value: 0.5104929304417525 and parameters: {'tree_method': 'gpu_hist', 'predictor': 'gpu_predictor', 'n_estimators': 129, 'max_depth': 5, 'learning_rate': 0.015379678837738699, 'subsample': 0.7407340960213598, 'colsample_bytree': 0.7269376565803354}. Best is trial 0 with value: 0.5104929304417525.


...

[I 2025-06-03 11:59:36,255] Trial 27 finished with value: 0.5718953572839989 and parameters: {'kernel': 'rbf', 'C': 6.072209836238589, 'epsilon': 0.04570681638474676}. Best is trial 13 with value: 0.5132354936695592.


Best trial: 13. Best value: 0.513235:  97%|█████████▋| 29/30 [6:24:42<12:16, 736.80s/it]

[I 2025-06-03 12:10:07,435] Trial 28 finished with value: 0.539033531605156 and parameters: {'kernel': 'rbf', 'C': 2.289322032947285, 'epsilon': 0.06533246336012644}. Best is trial 13 with value: 0.5132354936695592.


Best trial: 13. Best value: 0.513235: 100%|██████████| 30/30 [6:30:17<00:00, 780.60s/it]


[I 2025-06-03 12:15:42,692] Trial 29 finished with value: 0.520433076068974 and parameters: {'kernel': 'rbf', 'C': 0.9256896932279939, 'epsilon': 0.026342299558630745}. Best is trial 13 with value: 0.5132354936695592.
Support Vector Regressor Best Params: {'kernel': 'rbf', 'C': 0.33413350725146657, 'epsilon': 0.03419568430119482}

Model Evaluation Metrics:

----- XGBoost -----
RMSE: 0.646
MAE : 0.472
R2  : 53.12 %

----- LightGBM -----
RMSE: 0.646
MAE : 0.472
R2  : 53.03 %

----- Gradient Boosting -----
RMSE: 0.647
MAE : 0.472
R2  : 52.95 %

----- Decision Tree -----
RMSE: 0.655
MAE : 0.479
R2  : 51.67 %

----- Random Forest -----
RMSE: 0.646
MAE : 0.472
R2  : 53.02 %

----- Support Vector Regressor -----
RMSE: 0.663
MAE : 0.479
R2  : 50.52 %



***# Trial 2 : TrainTestSplit + lag_40 + rolling_std_3 + exp_avg_3***

In [30]:
# 2. Sort the dataframe by the full date (oldest to latest)
df2=df.copy()
df2 = df2.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df2['rolling_std'] = df2['daily_columno3'].shift(1).rolling(window=3).std()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day
df2['ema_avg'] = df2['daily_columno3'].shift(1).ewm(span=3, adjust=False).mean()


for i in range(1, 41):
    df2[f'lag{i}'] = df2['daily_columno3'].shift(i)


df2.dropna(inplace=True)
print(df2.shape)
df2.head()

(65470, 44)


,daily_date,daily_columno3,rolling_std,ema_avg,lag1,lag2,lag3,lag4,lag5,lag6,...,lag31,lag32,lag33,lag34,lag35,lag36,lag37,lag38,lag39,lag40
40,1976-10-06,275.7,10.161365,287.227325,278.6,296.2,296.2,291.3,301.1,307.9,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
41,1976-10-07,279.6,11.093692,281.463663,275.7,278.6,296.2,296.2,291.3,301.1,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
42,1976-10-08,274.8,2.025669,280.531831,279.6,275.7,278.6,296.2,296.2,291.3,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
43,1976-10-15,287.4,2.551470,277.665916,274.8,279.6,275.7,278.6,296.2,296.2,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
44,1976-10-18,271.8,6.359245,282.532958,287.4,274.8,279.6,275.7,278.6,296.2,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [31]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 41)]
lag_features.append('rolling_std')
lag_features.append('ema_avg')
X = df2[lag_features]
y = df2['daily_columno3']


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
X_train = minmax_x.fit_transform(X_train)
y_train = minmax_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
X_test = minmax_x.transform(X_test)
y_test = minmax_y.transform(y_test.values.reshape(-1, 1)).flatten()

# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results[name] = {
        'Model': model,
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001362 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10710
[LightGBM] [Info] Number of data points in the train set: 52376, number of used features: 42
[LightGBM] [Info] Start training from score 0.000000
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.647
MAE : 0.470
R2  : 52.94 %

----- Decision Tree -----
RMSE: 0.921
MAE : 0.668
R2  : 4.53 %

----- Random Forest -----
RMSE: 0.648
MAE : 0.472
R2  : 52.81 %

----- Gradient Boosting -----
RMSE: 0.649
MAE : 0.471
R2  : 52.66 %

----- Support Vector Regressor -----
RMSE: 0.668
MAE : 0.479
R2  : 49.81 %

----- XGBoost -----
RMSE: 0.666
MAE : 0.481
R2  : 50.10 %

----- LightGBM -----
RMSE: 0.649
MAE : 0.471
R2  : 52.58 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


***Tuning 2***

In [32]:

def tune_model(name, model_class, param_space):
    def objective(trial):
        params = {key: suggest(trial, key, dist) for key, dist in param_space.items()}
        model = model_class(**params)

        scores = []
        for train_idx, val_idx in cv.split(X_train):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            rmse = mean_squared_error(y_val, preds)
            scores.append(rmse)
        return np.mean(scores)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30, show_progress_bar=True)
    print(f"{name} Best Params: {study.best_params}")
    return model_class(**study.best_params)

def suggest(trial, name, dist):
    if isinstance(dist, tuple) and dist[0] == "int":
        return trial.suggest_int(name, *dist[1:])
    elif isinstance(dist, tuple) and dist[0] == "float":
        return trial.suggest_float(name, *dist[1:])
    elif isinstance(dist, tuple) and dist[0] == "cat":
        return trial.suggest_categorical(name, dist[1])
    else:
        raise ValueError(f"Unknown dist type for {name}: {dist}")

# Define parameter search spaces
param_spaces = {
    "XGBoost": {
        "tree_method": ("cat", ["gpu_hist"]),
        "predictor": ("cat", ["gpu_predictor"]),
        "n_estimators": ("int", 100, 150),
        "max_depth": ("int", 3, 7),
        "learning_rate": ("float", 0.01, 0.05),
        "subsample": ("float", 0.6, 0.9),
        "colsample_bytree": ("float", 0.6, 0.9)
    },
    "LightGBM": {  # Remove GPU
        "n_estimators": ("int", 100, 150),
        "learning_rate": ("float", 0.01, 0.05),
        "max_depth": ("int", 3, 7),
        "num_leaves": ("int", 15, 31),
        "subsample": ("float", 0.6, 0.9)
    },
    "Gradient Boosting": {
        "n_estimators": ("int", 100, 150),
        "learning_rate": ("float", 0.01, 0.1),
        "max_depth": ("int", 3, 5)
    },
    'Decision Tree': {
        'max_depth': ('int', 3, 10),
        'min_samples_split': ('int', 2, 10),
        'min_samples_leaf': ('int', 1, 5)
    },
    'Random Forest': {
        'n_estimators': ('int', 100, 200),
        'max_depth': ('int', 3, 10),
        'min_samples_split': ('int', 2, 10),
        'min_samples_leaf': ('int', 1, 5)
    },
    'Support Vector Regressor': {
        'kernel': ('cat', ['rbf']),
        'C': ('float', 0.1, 10.0),
        'epsilon': ('float', 0.01, 0.1)
    }
}

model_classes = {
    "XGBoost": XGBRegressor,
    "LightGBM": LGBMRegressor,
    "Gradient Boosting": GradientBoostingRegressor,
    'Decision Tree': DecisionTreeRegressor,
    'Random Forest': RandomForestRegressor,
    'Support Vector Regressor': SVR,
}

best_models = {}
results = {}

for name in model_classes:
    print(f"\nTuning {name}...")
    model = tune_model(name, model_classes[name], param_spaces[name])
    model.fit(X_train, y_train)
    best_models[name] = model

    # Evaluation
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# Print results
print("\nModel Evaluation Metrics:\n")
for name, metrics in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {metrics['RMSE']:.3f}")
    print(f"MAE : {metrics['MAE']:.3f}")
    print(f"R2  : {metrics['R2']*100:.2f} %\n")

[I 2025-06-03 12:25:04,801] A new study created in memory with name: no-name-64d3c845-7ef7-43b8-ae2a-3ae999a8b080



Tuning XGBoost...


Best trial: 0. Best value: 0.49282:   3%|▎         | 1/30 [00:01<00:45,  1.57s/it]

[I 2025-06-03 12:25:06,368] Trial 0 finished with value: 0.4928199908494657 and parameters: {'tree_method': 'gpu_hist', 'predictor': 'gpu_predictor', 'n_estimators': 141, 'max_depth': 6, 'learning_rate': 0.03012288105161453, 'subsample': 0.6297820001431801, 'colsample_bytree': 0.7458126630638575}. Best is trial 0 with value: 0.4928199908494657.


Best trial: 0. Best value: 0.49282:   7%|▋         | 2/30 [00:02<00:31,  1.14s/it]

...

[I 2025-06-03 18:31:48,177] Trial 27 finished with value: 0.5253603846813267 and parameters: {'kernel': 'rbf', 'C': 1.4976600924271086, 'epsilon': 0.04388560986046132}. Best is trial 21 with value: 0.5089066934888006.


Best trial: 21. Best value: 0.508907:  97%|█████████▋| 29/30 [3:41:52<05:51, 351.71s/it]

[I 2025-06-03 18:38:48,227] Trial 28 finished with value: 0.5372637940262808 and parameters: {'kernel': 'rbf', 'C': 2.474767385357592, 'epsilon': 0.019241699736107537}. Best is trial 21 with value: 0.5089066934888006.


Best trial: 21. Best value: 0.508907: 100%|██████████| 30/30 [3:44:48<00:00, 449.61s/it]


[I 2025-06-03 18:41:44,369] Trial 29 finished with value: 0.5153482734740568 and parameters: {'kernel': 'rbf', 'C': 0.7790210858129979, 'epsilon': 0.05955954551034308}. Best is trial 21 with value: 0.5089066934888006.
Support Vector Regressor Best Params: {'kernel': 'rbf', 'C': 0.21289772757171344, 'epsilon': 0.04511046339693563}

Model Evaluation Metrics:

----- XGBoost -----
RMSE: 0.647
MAE : 0.471
R2  : 52.94 %

----- LightGBM -----
RMSE: 0.647
MAE : 0.471
R2  : 52.83 %

----- Gradient Boosting -----
RMSE: 0.649
MAE : 0.471
R2  : 52.64 %

----- Decision Tree -----
RMSE: 0.678
MAE : 0.496
R2  : 48.20 %

----- Random Forest -----
RMSE: 0.649
MAE : 0.474
R2  : 52.57 %

----- Support Vector Regressor -----
RMSE: 0.661
MAE : 0.474
R2  : 50.82 %



**Trial 3 : TrainTestSplit + lag_90 + rolling_avg_14 + rolling_std_14 + exp_avg_14**

In [33]:
# 2. Sort the dataframe by the full date (oldest to latest)
df3=df.copy()
df3 = df3.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df3['rolling_std'] = df3['daily_columno3'].shift(1).rolling(window=14).std()
df3['rolling_avg'] = df3['daily_columno3'].shift(1).rolling(window=14).mean()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day
df3['ema_avg'] = df3['daily_columno3'].shift(1).ewm(span=14, adjust=False).mean()


for i in range(1, 91):
    df3[f'lag{i}'] = df3['daily_columno3'].shift(i)


df3.dropna(inplace=True)
print(df3.shape)
df3.head()

(65420, 95)


,daily_date,daily_columno3,rolling_std,rolling_avg,ema_avg,lag1,lag2,lag3,lag4,lag5,...,lag81,lag82,lag83,lag84,lag85,lag86,lag87,lag88,lag89,lag90
90,1980-06-04,337.0,23.701556,361.928571,356.964696,333.0,322.0,348.0,357.0,378.0,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
91,1980-06-06,359.0,24.624655,360.285714,354.302736,337.0,333.0,322.0,348.0,357.0,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
92,1980-06-09,332.0,24.183933,361.357143,354.929038,359.0,337.0,333.0,322.0,348.0,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
93,1980-06-10,336.0,25.180666,360.285714,351.871833,332.0,359.0,337.0,333.0,322.0,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
94,1980-06-11,356.0,25.973359,359.000000,349.755589,336.0,332.0,359.0,337.0,333.0,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [34]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 91)]
lag_features.append('rolling_avg')
lag_features.append('rolling_std')
lag_features.append('ema_avg')
X = df3[lag_features]
y = df3['daily_columno3']


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
X_train = minmax_x.fit_transform(X_train)
y_train = minmax_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
X_test = minmax_x.transform(X_test)
y_test = minmax_y.transform(y_test.values.reshape(-1, 1)).flatten()

# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results[name] = {
        'Model': model,
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003722 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 23715
[LightGBM] [Info] Number of data points in the train set: 52336, number of used features: 93
[LightGBM] [Info] Start training from score 0.000000
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.649
MAE : 0.472
R2  : 52.71 %

----- Decision Tree -----
RMSE: 0.915
MAE : 0.667
R2  : 5.97 %

----- Random Forest -----
RMSE: 0.648
MAE : 0.474
R2  : 52.79 %

----- Gradient Boosting -----
RMSE: 0.648
MAE : 0.473
R2  : 52.80 %

----- Support Vector Regressor -----
RMSE: 0.673
MAE : 0.485
R2  : 49.15 %

----- XGBoost -----
RMSE: 0.661
MAE : 0.479
R2  : 50.83 %

----- LightGBM -----
RMSE: 0.647
MAE : 0.472
R2  : 52.98 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


***Tuning 3***

In [35]:

def tune_model(name, model_class, param_space):
    def objective(trial):
        params = {key: suggest(trial, key, dist) for key, dist in param_space.items()}
        model = model_class(**params)

        scores = []
        for train_idx, val_idx in cv.split(X_train):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            rmse = mean_squared_error(y_val, preds)
            scores.append(rmse)
        return np.mean(scores)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30, show_progress_bar=True)
    print(f"{name} Best Params: {study.best_params}")
    return model_class(**study.best_params)

def suggest(trial, name, dist):
    if isinstance(dist, tuple) and dist[0] == "int":
        return trial.suggest_int(name, *dist[1:])
    elif isinstance(dist, tuple) and dist[0] == "float":
        return trial.suggest_float(name, *dist[1:])
    elif isinstance(dist, tuple) and dist[0] == "cat":
        return trial.suggest_categorical(name, dist[1])
    else:
        raise ValueError(f"Unknown dist type for {name}: {dist}")

# Define parameter search spaces
param_spaces = {
    "XGBoost": {
        "tree_method": ("cat", ["gpu_hist"]),
        "predictor": ("cat", ["gpu_predictor"]),
        "n_estimators": ("int", 100, 150),
        "max_depth": ("int", 3, 7),
        "learning_rate": ("float", 0.01, 0.05),
        "subsample": ("float", 0.6, 0.9),
        "colsample_bytree": ("float", 0.6, 0.9)
    },
    "LightGBM": {  # Remove GPU
        "n_estimators": ("int", 100, 150),
        "learning_rate": ("float", 0.01, 0.05),
        "max_depth": ("int", 3, 7),
        "num_leaves": ("int", 15, 31),
        "subsample": ("float", 0.6, 0.9)
    },
    "Gradient Boosting": {
        "n_estimators": ("int", 100, 150),
        "learning_rate": ("float", 0.01, 0.1),
        "max_depth": ("int", 3, 5)
    },
    'Decision Tree': {
        'max_depth': ('int', 3, 10),
        'min_samples_split': ('int', 2, 10),
        'min_samples_leaf': ('int', 1, 5)
    },
    'Random Forest': {
        'n_estimators': ('int', 100, 200),
        'max_depth': ('int', 3, 10),
        'min_samples_split': ('int', 2, 10),
        'min_samples_leaf': ('int', 1, 5)
    },
    'Support Vector Regressor': {
        'kernel': ('cat', ['rbf']),
        'C': ('float', 0.1, 10.0),
        'epsilon': ('float', 0.01, 0.1)
    }
}

model_classes = {
    "XGBoost": XGBRegressor,
    "LightGBM": LGBMRegressor,
    "Gradient Boosting": GradientBoostingRegressor,
    'Decision Tree': DecisionTreeRegressor,
    'Random Forest': RandomForestRegressor,
    'Support Vector Regressor': SVR,
}

best_models = {}
results = {}

for name in model_classes:
    print(f"\nTuning {name}...")
    model = tune_model(name, model_classes[name], param_spaces[name])
    model.fit(X_train, y_train)
    best_models[name] = model

    # Evaluation
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# Print results
print("\nModel Evaluation Metrics:\n")
for name, metrics in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {metrics['RMSE']:.3f}")
    print(f"MAE : {metrics['MAE']:.3f}")
    print(f"R2  : {metrics['R2']*100:.2f} %\n")

[I 2025-06-03 19:00:01,041] A new study created in memory with name: no-name-0528e968-3898-43f7-a170-3d46c0e8c834



Tuning XGBoost...


Best trial: 0. Best value: 0.495435:   3%|▎         | 1/30 [00:01<00:41,  1.43s/it]

[I 2025-06-03 19:00:02,469] Trial 0 finished with value: 0.4954345502074487 and parameters: {'tree_method': 'gpu_hist', 'predictor': 'gpu_predictor', 'n_estimators': 134, 'max_depth': 5, 'learning_rate': 0.03004771764333447, 'subsample': 0.8631643168741668, 'colsample_bytree': 0.6619813148046421}. Best is trial 0 with value: 0.4954345502074487.


Best trial: 0. Best value: 0.495435:   7%|▋         | 2/30 [00:03<00:53,  1.90s/it]

...

[I 2025-06-04 08:54:16,361] Trial 27 finished with value: 0.5346370304346256 and parameters: {'kernel': 'rbf', 'C': 1.7798275589525805, 'epsilon': 0.0712067426176879}. Best is trial 3 with value: 0.5142494753936849.


Best trial: 3. Best value: 0.514249:  97%|█████████▋| 29/30 [8:34:20<10:18, 618.57s/it]

[I 2025-06-04 09:02:34,853] Trial 28 finished with value: 0.5218367509100379 and parameters: {'kernel': 'rbf', 'C': 0.9241047882833744, 'epsilon': 0.04808316709257525}. Best is trial 3 with value: 0.5142494753936849.


Best trial: 3. Best value: 0.514249: 100%|██████████| 30/30 [9:00:12<00:00, 1080.43s/it]


[I 2025-06-04 09:28:27,197] Trial 29 finished with value: 0.5634724999990013 and parameters: {'kernel': 'rbf', 'C': 4.077635129448415, 'epsilon': 0.031899807926039525}. Best is trial 3 with value: 0.5142494753936849.
Support Vector Regressor Best Params: {'kernel': 'rbf', 'C': 0.30663642610715847, 'epsilon': 0.07209954654498156}

Model Evaluation Metrics:

----- XGBoost -----
RMSE: 0.645
MAE : 0.471
R2  : 53.25 %

----- LightGBM -----
RMSE: 0.645
MAE : 0.471
R2  : 53.18 %

----- Gradient Boosting -----
RMSE: 0.646
MAE : 0.472
R2  : 53.05 %

----- Decision Tree -----
RMSE: 0.653
MAE : 0.477
R2  : 52.08 %

----- Random Forest -----
RMSE: 0.646
MAE : 0.472
R2  : 53.16 %

----- Support Vector Regressor -----
RMSE: 0.667
MAE : 0.480
R2  : 49.99 %



**Trial 4 : TrainTestSplit + lag_70 + rolling_avg_7**

In [36]:
# 2. Sort the dataframe by the full date (oldest to latest)
df4=df.copy()
df4 = df4.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df4['rolling_avg'] = df4['daily_columno3'].shift(1).rolling(window=7).mean()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day


for i in range(1, 71):
    df4[f'lag{i}'] = df4['daily_columno3'].shift(i)


df4.dropna(inplace=True)
print(df4.shape)
df4.head()

(65440, 73)


,daily_date,daily_columno3,rolling_avg,lag1,lag2,lag3,lag4,lag5,lag6,lag7,...,lag61,lag62,lag63,lag64,lag65,lag66,lag67,lag68,lag69,lag70
70,1980-04-11,344.0,350.857143,329.0,354.0,333.0,381.0,363.0,373.0,323.0,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
71,1980-04-14,360.0,353.857143,344.0,329.0,354.0,333.0,381.0,363.0,373.0,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
72,1980-04-15,366.0,352.000000,360.0,344.0,329.0,354.0,333.0,381.0,363.0,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
73,1980-04-16,398.0,352.428571,366.0,360.0,344.0,329.0,354.0,333.0,381.0,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
74,1980-04-17,393.0,354.857143,398.0,366.0,360.0,344.0,329.0,354.0,333.0,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [37]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 71)]
lag_features.append('rolling_avg')
X = df4[lag_features]
y = df4['daily_columno3']


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
X_train = minmax_x.fit_transform(X_train)
y_train = minmax_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
X_test = minmax_x.transform(X_test)
y_test = minmax_y.transform(y_test.values.reshape(-1, 1)).flatten()

# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results[name] = {
        'Model': model,
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003164 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 18105
[LightGBM] [Info] Number of data points in the train set: 52352, number of used features: 71
[LightGBM] [Info] Start training from score 0.000000
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.648
MAE : 0.472
R2  : 52.69 %

----- Decision Tree -----
RMSE: 0.938
MAE : 0.679
R2  : 1.02 %

----- Random Forest -----
RMSE: 0.648
MAE : 0.474
R2  : 52.73 %

----- Gradient Boosting -----
RMSE: 0.649
MAE : 0.472
R2  : 52.57 %

----- Support Vector Regressor -----
RMSE: 0.669
MAE : 0.484
R2  : 49.63 %

----- XGBoost -----
RMSE: 0.661
MAE : 0.480
R2  : 50.78 %

----- LightGBM -----
RMSE: 0.647
MAE : 0.472
R2  : 52.85 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


***Tuning 4***

In [38]:

def tune_model(name, model_class, param_space):
    def objective(trial):
        params = {key: suggest(trial, key, dist) for key, dist in param_space.items()}
        model = model_class(**params)

        scores = []
        for train_idx, val_idx in cv.split(X_train):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            rmse = mean_squared_error(y_val, preds)
            scores.append(rmse)
        return np.mean(scores)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30, show_progress_bar=True)
    print(f"{name} Best Params: {study.best_params}")
    return model_class(**study.best_params)

def suggest(trial, name, dist):
    if isinstance(dist, tuple) and dist[0] == "int":
        return trial.suggest_int(name, *dist[1:])
    elif isinstance(dist, tuple) and dist[0] == "float":
        return trial.suggest_float(name, *dist[1:])
    elif isinstance(dist, tuple) and dist[0] == "cat":
        return trial.suggest_categorical(name, dist[1])
    else:
        raise ValueError(f"Unknown dist type for {name}: {dist}")

# Define parameter search spaces
param_spaces = {
    "XGBoost": {
        "tree_method": ("cat", ["gpu_hist"]),
        "predictor": ("cat", ["gpu_predictor"]),
        "n_estimators": ("int", 100, 150),
        "max_depth": ("int", 3, 7),
        "learning_rate": ("float", 0.01, 0.05),
        "subsample": ("float", 0.6, 0.9),
        "colsample_bytree": ("float", 0.6, 0.9)
    },
    "LightGBM": { 
        "n_estimators": ("int", 100, 150),
        "learning_rate": ("float", 0.01, 0.05),
        "max_depth": ("int", 3, 7),
        "num_leaves": ("int", 15, 31),
        "subsample": ("float", 0.6, 0.9)
    },
    "Gradient Boosting": {
        "n_estimators": ("int", 100, 150),
        "learning_rate": ("float", 0.01, 0.1),
        "max_depth": ("int", 3, 5)
    },
    'Decision Tree': {
        'max_depth': ('int', 3, 10),
        'min_samples_split': ('int', 2, 10),
        'min_samples_leaf': ('int', 1, 5)
    },
    'Random Forest': {
        'n_estimators': ('int', 100, 200),
        'max_depth': ('int', 3, 10),
        'min_samples_split': ('int', 2, 10),
        'min_samples_leaf': ('int', 1, 5)
    },
    'Support Vector Regressor': {
        'kernel': ('cat', ['rbf']),
        'C': ('float', 0.1, 10.0),
        'epsilon': ('float', 0.01, 0.1)
    }
}

model_classes = {
    "XGBoost": XGBRegressor,
    "LightGBM": LGBMRegressor,
    "Gradient Boosting": GradientBoostingRegressor,
    'Decision Tree': DecisionTreeRegressor,
    'Random Forest': RandomForestRegressor,
    'Support Vector Regressor': SVR,
}

best_models = {}
results = {}

for name in model_classes:
    print(f"\nTuning {name}...")
    model = tune_model(name, model_classes[name], param_spaces[name])
    model.fit(X_train, y_train)
    best_models[name] = model

    # Evaluation
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# Print results
print("\nModel Evaluation Metrics:\n")
for name, metrics in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {metrics['RMSE']:.3f}")
    print(f"MAE : {metrics['MAE']:.3f}")
    print(f"R2  : {metrics['R2']*100:.2f} %\n")

[I 2025-06-04 09:43:59,060] A new study created in memory with name: no-name-995e2b39-ba50-4502-b972-2f11969a192d



Tuning XGBoost...


Best trial: 0. Best value: 0.496781:   3%|▎         | 1/30 [00:00<00:28,  1.00it/s]

[I 2025-06-04 09:44:00,054] Trial 0 finished with value: 0.49678104648280935 and parameters: {'tree_method': 'gpu_hist', 'predictor': 'gpu_predictor', 'n_estimators': 131, 'max_depth': 3, 'learning_rate': 0.04082272171848409, 'subsample': 0.6526944986962466, 'colsample_bytree': 0.8211042472565548}. Best is trial 0 with value: 0.49678104648280935.


Best trial: 1. Best value: 0.494142:   7%|▋         | 2/30 [00:01<00:24,  1.13it/s]

...

[I 2025-06-04 20:06:35,926] Trial 27 finished with value: 0.5162065448679719 and parameters: {'kernel': 'rbf', 'C': 0.17159511054597132, 'epsilon': 0.0466988716309528}. Best is trial 22 with value: 0.5141590954743874.


Best trial: 22. Best value: 0.514159:  97%|█████████▋| 29/30 [6:22:54<10:03, 603.36s/it]

[I 2025-06-04 20:15:28,181] Trial 28 finished with value: 0.5348820629960278 and parameters: {'kernel': 'rbf', 'C': 1.9268659262349206, 'epsilon': 0.06269716613395807}. Best is trial 22 with value: 0.5141590954743874.


Best trial: 22. Best value: 0.514159: 100%|██████████| 30/30 [6:36:36<00:00, 793.21s/it]


[I 2025-06-04 20:29:10,142] Trial 29 finished with value: 0.5495525889525016 and parameters: {'kernel': 'rbf', 'C': 3.066686538843838, 'epsilon': 0.026821743502114907}. Best is trial 22 with value: 0.5141590954743874.
Support Vector Regressor Best Params: {'kernel': 'rbf', 'C': 0.2822678841592354, 'epsilon': 0.033565886604085576}

Model Evaluation Metrics:

----- XGBoost -----
RMSE: 0.645
MAE : 0.470
R2  : 53.14 %

----- LightGBM -----
RMSE: 0.646
MAE : 0.471
R2  : 53.03 %

----- Gradient Boosting -----
RMSE: 0.646
MAE : 0.471
R2  : 53.04 %

----- Decision Tree -----
RMSE: 0.657
MAE : 0.479
R2  : 51.51 %

----- Random Forest -----
RMSE: 0.646
MAE : 0.472
R2  : 52.98 %

----- Support Vector Regressor -----
RMSE: 0.663
MAE : 0.479
R2  : 50.51 %



**Trial 5 : TrainTestSplit + lag_100 + rolling_avg_7**

In [39]:
# 2. Sort the dataframe by the full date (oldest to latest)
df5=df.copy()
df5 = df5.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df5['rolling_avg'] = df5['daily_columno3'].shift(1).rolling(window=7).mean()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day


for i in range(1, 101):
    df5[f'lag{i}'] = df5['daily_columno3'].shift(i)


df5.dropna(inplace=True)
print(df5.shape)
df5.head()

(65410, 103)


/tmp/ipykernel_294262/2606774194.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df5[f'lag{i}'] = df5['daily_columno3'].shift(i)
/tmp/ipykernel_294262/2606774194.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df5[f'lag{i}'] = df5['daily_columno3'].shift(i)
/tmp/ipykernel_294262/2606774194.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a 

,daily_date,daily_columno3,rolling_avg,lag1,lag2,lag3,lag4,lag5,lag6,lag7,...,lag91,lag92,lag93,lag94,lag95,lag96,lag97,lag98,lag99,lag100
100,1980-07-01,333.0,342.714286,321.0,331.0,349.0,340.0,366.0,356.0,336.0,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
101,1980-07-02,317.0,342.285714,333.0,321.0,331.0,349.0,340.0,366.0,356.0,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
102,1980-07-03,322.0,336.714286,317.0,333.0,321.0,331.0,349.0,340.0,366.0,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
103,1980-07-04,323.0,330.428571,322.0,317.0,333.0,321.0,331.0,349.0,340.0,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
104,1980-07-07,327.0,328.000000,323.0,322.0,317.0,333.0,321.0,331.0,349.0,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [40]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 101)]
lag_features.append('rolling_avg')
X = df5[lag_features]
y = df5['daily_columno3']


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
X_train = minmax_x.fit_transform(X_train)
y_train = minmax_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
X_test = minmax_x.transform(X_test)
y_test = minmax_y.transform(y_test.values.reshape(-1, 1)).flatten()

# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results[name] = {
        'Model': model,
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004044 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25755
[LightGBM] [Info] Number of data points in the train set: 52328, number of used features: 101
[LightGBM] [Info] Start training from score -0.000000
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.649
MAE : 0.473
R2  : 52.66 %

----- Decision Tree -----
RMSE: 0.926
MAE : 0.671
R2  : 3.69 %

----- Random Forest -----
RMSE: 0.648
MAE : 0.474
R2  : 52.81 %

----- Gradient Boosting -----
RMSE: 0.649
MAE : 0.472
R2  : 52.61 %

----- Support Vector Regressor -----
RMSE: 0.674
MAE : 0.487
R2  : 49.00 %

----- XGBoost -----
RMSE: 0.664
MAE : 0.480
R2  : 50.47 %

----- LightGBM -----
RMSE: 0.649
MAE : 0.472
R2  : 52.73 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


***Tuning 5***

In [41]:

def tune_model(name, model_class, param_space):
    def objective(trial):
        params = {key: suggest(trial, key, dist) for key, dist in param_space.items()}
        model = model_class(**params)

        scores = []
        for train_idx, val_idx in cv.split(X_train):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            rmse = mean_squared_error(y_val, preds)
            scores.append(rmse)
        return np.mean(scores)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30, show_progress_bar=True)
    print(f"{name} Best Params: {study.best_params}")
    return model_class(**study.best_params)

def suggest(trial, name, dist):
    if isinstance(dist, tuple) and dist[0] == "int":
        return trial.suggest_int(name, *dist[1:])
    elif isinstance(dist, tuple) and dist[0] == "float":
        return trial.suggest_float(name, *dist[1:])
    elif isinstance(dist, tuple) and dist[0] == "cat":
        return trial.suggest_categorical(name, dist[1])
    else:
        raise ValueError(f"Unknown dist type for {name}: {dist}")

# Define parameter search spaces
param_spaces = {
    "XGBoost": {
        "tree_method": ("cat", ["gpu_hist"]),
        "predictor": ("cat", ["gpu_predictor"]),
        "n_estimators": ("int", 100, 150),
        "max_depth": ("int", 3, 7),
        "learning_rate": ("float", 0.01, 0.05),
        "subsample": ("float", 0.6, 0.9),
        "colsample_bytree": ("float", 0.6, 0.9)
    },
    "LightGBM": {  # Remove GPU
        "n_estimators": ("int", 100, 150),
        "learning_rate": ("float", 0.01, 0.05),
        "max_depth": ("int", 3, 7),
        "num_leaves": ("int", 15, 31),
        "subsample": ("float", 0.6, 0.9)
    },
    "Gradient Boosting": {
        "n_estimators": ("int", 100, 150),
        "learning_rate": ("float", 0.01, 0.1),
        "max_depth": ("int", 3, 5)
    },
    'Decision Tree': {
        'max_depth': ('int', 3, 10),
        'min_samples_split': ('int', 2, 10),
        'min_samples_leaf': ('int', 1, 5)
    },
    'Random Forest': {
        'n_estimators': ('int', 100, 200),
        'max_depth': ('int', 3, 10),
        'min_samples_split': ('int', 2, 10),
        'min_samples_leaf': ('int', 1, 5)
    },
    'Support Vector Regressor': {
        'kernel': ('cat', ['rbf']),
        'C': ('float', 0.1, 10.0),
        'epsilon': ('float', 0.01, 0.1)
    }
}

model_classes = {
    "XGBoost": XGBRegressor,
    "LightGBM": LGBMRegressor,
    "Gradient Boosting": GradientBoostingRegressor,
    'Decision Tree': DecisionTreeRegressor,
    'Random Forest': RandomForestRegressor,
    'Support Vector Regressor': SVR,
}

best_models = {}
results = {}

for name in model_classes:
    print(f"\nTuning {name}...")
    model = tune_model(name, model_classes[name], param_spaces[name])
    model.fit(X_train, y_train)
    best_models[name] = model

    # Evaluation
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# Print results
print("\nModel Evaluation Metrics:\n")
for name, metrics in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {metrics['RMSE']:.3f}")
    print(f"MAE : {metrics['MAE']:.3f}")
    print(f"R2  : {metrics['R2']*100:.2f} %\n")

[I 2025-06-04 20:49:01,371] A new study created in memory with name: no-name-3589366a-07c1-4dd8-b2d6-7fcb515337f2



Tuning XGBoost...


Best trial: 0. Best value: 0.52461:   3%|▎         | 1/30 [00:01<00:42,  1.46s/it]

[I 2025-06-04 20:49:02,830] Trial 0 finished with value: 0.5246097889732341 and parameters: {'tree_method': 'gpu_hist', 'predictor': 'gpu_predictor', 'n_estimators': 112, 'max_depth': 4, 'learning_rate': 0.01563842503748724, 'subsample': 0.8637564148301673, 'colsample_bytree': 0.8129286353190819}. Best is trial 0 with value: 0.5246097889732341.


Best trial: 1. Best value: 0.498086:   7%|▋         | 2/30 [00:03<00:52,  1.89s/it]

...

[I 2025-06-05 12:14:21,854] Trial 27 finished with value: 0.5266072198440677 and parameters: {'kernel': 'rbf', 'C': 1.2912937248587122, 'epsilon': 0.05127711427730725}. Best is trial 21 with value: 0.5160313557497395.


Best trial: 21. Best value: 0.516031:  97%|█████████▋| 29/30 [9:37:14<11:06, 666.79s/it]

[I 2025-06-05 12:19:01,788] Trial 28 finished with value: 0.5161967455742599 and parameters: {'kernel': 'rbf', 'C': 0.2740915518328536, 'epsilon': 0.07900755423192786}. Best is trial 21 with value: 0.5160313557497395.


Best trial: 21. Best value: 0.516031: 100%|██████████| 30/30 [9:45:35<00:00, 1171.17s/it]


[I 2025-06-05 12:27:22,797] Trial 29 finished with value: 0.519796295384323 and parameters: {'kernel': 'rbf', 'C': 0.8480627116117568, 'epsilon': 0.0800633519900916}. Best is trial 21 with value: 0.5160313557497395.
Support Vector Regressor Best Params: {'kernel': 'rbf', 'C': 0.301332190074046, 'epsilon': 0.07073485225165388}

Model Evaluation Metrics:

----- XGBoost -----
RMSE: 0.645
MAE : 0.471
R2  : 53.22 %

----- LightGBM -----
RMSE: 0.647
MAE : 0.472
R2  : 53.00 %

----- Gradient Boosting -----
RMSE: 0.646
MAE : 0.472
R2  : 53.05 %

----- Decision Tree -----
RMSE: 0.657
MAE : 0.478
R2  : 51.43 %

----- Random Forest -----
RMSE: 0.647
MAE : 0.472
R2  : 52.96 %

----- Support Vector Regressor -----
RMSE: 0.668
MAE : 0.482
R2  : 49.78 %



**Trial 6 : TrainTestSplit + lag_50 + exp_avg_14**

In [42]:
# 2. Sort the dataframe by the full date (oldest to latest)
df6=df.copy()
df6 = df6.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df6['ema_avg'] = df6['daily_columno3'].shift(1).ewm(span=14, adjust=False).mean()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day


for i in range(1, 51):
    df6[f'lag{i}'] = df6['daily_columno3'].shift(i)


df6.dropna(inplace=True)
print(df6.shape)
df6.head()

(65460, 53)


,daily_date,daily_columno3,ema_avg,lag1,lag2,lag3,lag4,lag5,lag6,lag7,...,lag41,lag42,lag43,lag44,lag45,lag46,lag47,lag48,lag49,lag50
50,1976-11-17,281.6,283.560651,295.2,305.9,267.9,267.0,262.1,271.8,287.4,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
51,1976-11-18,270.9,283.299231,281.6,295.2,305.9,267.9,267.0,262.1,271.8,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
52,1976-11-19,290.3,281.646000,270.9,281.6,295.2,305.9,267.9,267.0,262.1,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
53,1976-11-22,296.2,282.799867,290.3,270.9,281.6,295.2,305.9,267.9,267.0,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
54,1976-11-23,267.9,284.586551,296.2,290.3,270.9,281.6,295.2,305.9,267.9,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [43]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 51)]
lag_features.append('ema_avg')
X = df6[lag_features]
y = df6['daily_columno3']


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
X_train = minmax_x.fit_transform(X_train)
y_train = minmax_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
X_test = minmax_x.transform(X_test)
y_test = minmax_y.transform(y_test.values.reshape(-1, 1)).flatten()

# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results[name] = {
        'Model': model,
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")



[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002093 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13005
[LightGBM] [Info] Number of data points in the train set: 52368, number of used features: 51
[LightGBM] [Info] Start training from score 0.000000
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.648
MAE : 0.472
R2  : 52.72 %

----- Decision Tree -----
RMSE: 0.904
MAE : 0.658
R2  : 8.04 %

----- Random Forest -----
RMSE: 0.650
MAE : 0.475
R2  : 52.52 %

----- Gradient Boosting -----
RMSE: 0.649
MAE : 0.473
R2  : 52.67 %

----- Support Vector Regressor -----
RMSE: 0.668
MAE : 0.482
R2  : 49.78 %

----- XGBoost -----
RMSE: 0.664
MAE : 0.481
R2  : 50.38 %

----- LightGBM -----
RMSE: 0.647
MAE : 0.471
R2  : 52.90 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


***Tuining 6**

In [44]:

def tune_model(name, model_class, param_space):
    def objective(trial):
        params = {key: suggest(trial, key, dist) for key, dist in param_space.items()}
        model = model_class(**params)

        scores = []
        for train_idx, val_idx in cv.split(X_train):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            rmse = mean_squared_error(y_val, preds)
            scores.append(rmse)
        return np.mean(scores)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30, show_progress_bar=True)
    print(f"{name} Best Params: {study.best_params}")
    return model_class(**study.best_params)

def suggest(trial, name, dist):
    if isinstance(dist, tuple) and dist[0] == "int":
        return trial.suggest_int(name, *dist[1:])
    elif isinstance(dist, tuple) and dist[0] == "float":
        return trial.suggest_float(name, *dist[1:])
    elif isinstance(dist, tuple) and dist[0] == "cat":
        return trial.suggest_categorical(name, dist[1])
    else:
        raise ValueError(f"Unknown dist type for {name}: {dist}")

# Define parameter search spaces
param_spaces = {
    "XGBoost": {
        "tree_method": ("cat", ["gpu_hist"]),
        "predictor": ("cat", ["gpu_predictor"]),
        "n_estimators": ("int", 100, 150),
        "max_depth": ("int", 3, 7),
        "learning_rate": ("float", 0.01, 0.05),
        "subsample": ("float", 0.6, 0.9),
        "colsample_bytree": ("float", 0.6, 0.9)
    },
    "LightGBM": {  # Remove GPU
        "n_estimators": ("int", 100, 150),
        "learning_rate": ("float", 0.01, 0.05),
        "max_depth": ("int", 3, 7),
        "num_leaves": ("int", 15, 31),
        "subsample": ("float", 0.6, 0.9)
    },
    "Gradient Boosting": {
        "n_estimators": ("int", 100, 150),
        "learning_rate": ("float", 0.01, 0.1),
        "max_depth": ("int", 3, 5)
    },
    'Decision Tree': {
        'max_depth': ('int', 3, 10),
        'min_samples_split': ('int', 2, 10),
        'min_samples_leaf': ('int', 1, 5)
    },
    'Random Forest': {
        'n_estimators': ('int', 100, 200),
        'max_depth': ('int', 3, 10),
        'min_samples_split': ('int', 2, 10),
        'min_samples_leaf': ('int', 1, 5)
    },
    'Support Vector Regressor': {
        'kernel': ('cat', ['rbf']),
        'C': ('float', 0.1, 10.0),
        'epsilon': ('float', 0.01, 0.1)
    }
}

model_classes = {
    "XGBoost": XGBRegressor,
    "LightGBM": LGBMRegressor,
    "Gradient Boosting": GradientBoostingRegressor,
    'Decision Tree': DecisionTreeRegressor,
    'Random Forest': RandomForestRegressor,
    'Support Vector Regressor': SVR,
}

best_models = {}
results = {}

for name in model_classes:
    print(f"\nTuning {name}...")
    model = tune_model(name, model_classes[name], param_spaces[name])
    model.fit(X_train, y_train)
    best_models[name] = model

    # Evaluation
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# Print results
print("\nModel Evaluation Metrics:\n")
for name, metrics in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {metrics['RMSE']:.3f}")
    print(f"MAE : {metrics['MAE']:.3f}")
    print(f"R2  : {metrics['R2']*100:.2f} %\n")

[I 2025-06-05 12:39:35,713] A new study created in memory with name: no-name-00af34be-58ee-4cfa-8325-00bc144e47dc



Tuning XGBoost...


Best trial: 0. Best value: 0.499846:   3%|▎         | 1/30 [00:02<00:58,  2.03s/it]

[I 2025-06-05 12:39:37,746] Trial 0 finished with value: 0.4998457537991041 and parameters: {'tree_method': 'gpu_hist', 'predictor': 'gpu_predictor', 'n_estimators': 130, 'max_depth': 7, 'learning_rate': 0.01992098344053162, 'subsample': 0.8526507318687492, 'colsample_bytree': 0.7610778096364976}. Best is trial 0 with value: 0.4998457537991041.


Best trial: 1. Best value: 0.498279:   7%|▋         | 2/30 [00:02<00:34,  1.24s/it]

...

[I 2025-06-05 19:09:14,087] Trial 27 finished with value: 0.5196574940996762 and parameters: {'kernel': 'rbf', 'C': 0.9156075208975969, 'epsilon': 0.06334435395590177}. Best is trial 19 with value: 0.5122739579900628.


Best trial: 19. Best value: 0.512274:  97%|█████████▋| 29/30 [3:49:02<05:00, 300.08s/it]

[I 2025-06-05 19:15:02,127] Trial 28 finished with value: 0.5331415688912117 and parameters: {'kernel': 'rbf', 'C': 1.835037586648623, 'epsilon': 0.034043490413846154}. Best is trial 19 with value: 0.5122739579900628.


Best trial: 19. Best value: 0.512274: 100%|██████████| 30/30 [3:58:06<00:00, 476.22s/it]


[I 2025-06-05 19:24:06,531] Trial 29 finished with value: 0.5479870507442612 and parameters: {'kernel': 'rbf', 'C': 3.1156955600257645, 'epsilon': 0.01949681869054034}. Best is trial 19 with value: 0.5122739579900628.
Support Vector Regressor Best Params: {'kernel': 'rbf', 'C': 0.28210185793627757, 'epsilon': 0.0357818322366628}

Model Evaluation Metrics:

----- XGBoost -----
RMSE: 0.645
MAE : 0.471
R2  : 53.21 %

----- LightGBM -----
RMSE: 0.645
MAE : 0.471
R2  : 53.16 %

----- Gradient Boosting -----
RMSE: 0.645
MAE : 0.472
R2  : 53.13 %

----- Decision Tree -----
RMSE: 0.652
MAE : 0.477
R2  : 52.09 %

----- Random Forest -----
RMSE: 0.645
MAE : 0.471
R2  : 53.18 %

----- Support Vector Regressor -----
RMSE: 0.661
MAE : 0.477
R2  : 50.78 %



**Trial 7 : TrainTestSplit + lag_60 + rolling_std_15**

In [45]:
# 2. Sort the dataframe by the full date (oldest to latest)
df7=df.copy()
df7 = df7.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df7['rolling_std'] = df7['daily_columno3'].shift(1).rolling(window=15).std()


for i in range(1, 61):
    df7[f'lag{i}'] = df7['daily_columno3'].shift(i)


df7.dropna(inplace=True)
print(df7.shape)
df7.head()

(65450, 63)


,daily_date,daily_columno3,rolling_std,lag1,lag2,lag3,lag4,lag5,lag6,lag7,...,lag51,lag52,lag53,lag54,lag55,lag56,lag57,lag58,lag59,lag60
60,1980-02-26,340.0,15.337992,286.0,287.4,297.2,311.8,294.2,267.9,296.2,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
61,1980-02-27,308.0,19.495482,340.0,286.0,287.4,297.2,311.8,294.2,267.9,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
62,1980-03-05,424.0,18.807357,308.0,340.0,286.0,287.4,297.2,311.8,294.2,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
63,1980-03-06,323.0,37.553837,424.0,308.0,340.0,286.0,287.4,297.2,311.8,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
64,1980-03-12,373.0,37.881127,323.0,424.0,308.0,340.0,286.0,287.4,297.2,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [46]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 61)]
lag_features.append('rolling_std')
X = df7[lag_features]
y = df7['daily_columno3']


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
X_train = minmax_x.fit_transform(X_train)
y_train = minmax_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
X_test = minmax_x.transform(X_test)
y_test = minmax_y.transform(y_test.values.reshape(-1, 1)).flatten()

# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results[name] = {
        'Model': model,
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002628 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 52360, number of used features: 61
[LightGBM] [Info] Start training from score -0.000000
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.648
MAE : 0.472
R2  : 52.71 %

----- Decision Tree -----
RMSE: 0.938
MAE : 0.676
R2  : 1.06 %

----- Random Forest -----
RMSE: 0.648
MAE : 0.472
R2  : 52.73 %

----- Gradient Boosting -----
RMSE: 0.651
MAE : 0.472
R2  : 52.30 %

----- Support Vector Regressor -----
RMSE: 0.669
MAE : 0.483
R2  : 49.57 %

----- XGBoost -----
RMSE: 0.667
MAE : 0.480
R2  : 49.90 %

----- LightGBM -----
RMSE: 0.651
MAE : 0.471
R2  : 52.34 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


**Tuning 7**

In [ ]:

def tune_model(name, model_class, param_space):
    def objective(trial):
        params = {key: suggest(trial, key, dist) for key, dist in param_space.items()}
        model = model_class(**params)

        scores = []
        for train_idx, val_idx in cv.split(X_train):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            rmse = mean_squared_error(y_val, preds)
            scores.append(rmse)
        return np.mean(scores)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30, show_progress_bar=True)
    print(f"{name} Best Params: {study.best_params}")
    return model_class(**study.best_params)

def suggest(trial, name, dist):
    if isinstance(dist, tuple) and dist[0] == "int":
        return trial.suggest_int(name, *dist[1:])
    elif isinstance(dist, tuple) and dist[0] == "float":
        return trial.suggest_float(name, *dist[1:])
    elif isinstance(dist, tuple) and dist[0] == "cat":
        return trial.suggest_categorical(name, dist[1])
    else:
        raise ValueError(f"Unknown dist type for {name}: {dist}")

# Define parameter search spaces
param_spaces = {
    "XGBoost": {
        "tree_method": ("cat", ["gpu_hist"]),
        "predictor": ("cat", ["gpu_predictor"]),
        "n_estimators": ("int", 100, 150),
        "max_depth": ("int", 3, 7),
        "learning_rate": ("float", 0.01, 0.05),
        "subsample": ("float", 0.6, 0.9),
        "colsample_bytree": ("float", 0.6, 0.9)
    },
    "LightGBM": {  # Remove GPU
        "n_estimators": ("int", 100, 150),
        "learning_rate": ("float", 0.01, 0.05),
        "max_depth": ("int", 3, 7),
        "num_leaves": ("int", 15, 31),
        "subsample": ("float", 0.6, 0.9)
    },
    "Gradient Boosting": {
        "n_estimators": ("int", 100, 150),
        "learning_rate": ("float", 0.01, 0.1),
        "max_depth": ("int", 3, 5)
    },
    'Decision Tree': {
        'max_depth': ('int', 3, 10),
        'min_samples_split': ('int', 2, 10),
        'min_samples_leaf': ('int', 1, 5)
    },
    'Random Forest': {
        'n_estimators': ('int', 100, 200),
        'max_depth': ('int', 3, 10),
        'min_samples_split': ('int', 2, 10),
        'min_samples_leaf': ('int', 1, 5)
    },
    'Support Vector Regressor': {
        'kernel': ('cat', ['rbf']),
        'C': ('float', 0.1, 10.0),
        'epsilon': ('float', 0.01, 0.1)
    }
}

model_classes = {
    "XGBoost": XGBRegressor,
    "LightGBM": LGBMRegressor,
    "Gradient Boosting": GradientBoostingRegressor,
    'Decision Tree': DecisionTreeRegressor,
    'Random Forest': RandomForestRegressor,
    'Support Vector Regressor': SVR,
}

best_models = {}
results = {}

for name in model_classes:
    print(f"\nTuning {name}...")
    model = tune_model(name, model_classes[name], param_spaces[name])
    model.fit(X_train, y_train)
    best_models[name] = model

    # Evaluation
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# Print results
print("\nModel Evaluation Metrics:\n")
for name, metrics in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {metrics['RMSE']:.3f}")
    print(f"MAE : {metrics['MAE']:.3f}")
    print(f"R2  : {metrics['R2']*100:.2f} %\n")

[I 2025-06-05 19:35:09,019] A new study created in memory with name: no-name-caf5a139-b3bb-4108-a059-9e706e231b9c



Tuning XGBoost...


Best trial: 0. Best value: 0.499634:   3%|▎         | 1/30 [00:01<00:34,  1.18s/it]

[I 2025-06-05 19:35:10,195] Trial 0 finished with value: 0.4996337880150197 and parameters: {'tree_method': 'gpu_hist', 'predictor': 'gpu_predictor', 'n_estimators': 109, 'max_depth': 5, 'learning_rate': 0.025124613145293715, 'subsample': 0.7438073267753795, 'colsample_bytree': 0.6816634704666389}. Best is trial 0 with value: 0.4996337880150197.


Best trial: 1. Best value: 0.495449:   7%|▋         | 2/30 [00:02<00:28,  1.01s/it]

...

[I 2025-06-05 21:47:56,807] Trial 14 finished with value: 0.4952652838148051 and parameters: {'n_estimators': 181, 'max_depth': 10, 'min_samples_split': 4, 'min_samples_leaf': 4}. Best is trial 3 with value: 0.4949440515359768.


Best trial: 3. Best value: 0.494944:  53%|█████▎    | 16/30 [56:54<1:07:14, 288.17s/it]

[I 2025-06-05 21:52:28,994] Trial 15 finished with value: 0.49858214525204514 and parameters: {'n_estimators': 165, 'max_depth': 8, 'min_samples_split': 7, 'min_samples_leaf': 2}. Best is trial 3 with value: 0.4949440515359768.


Best trial: 3. Best value: 0.494944:  57%|█████▋    | 17/30 [1:02:37<1:05:58, 304.51s/it]

[I 2025-06-05 21:58:11,509] Trial 16 finished with value: 0.49634059398997127 and parameters: {'n_estimators': 189, 'max_depth': 9, 'min_samples_split': 5, 'min_samples_leaf': 4}. Best is trial 3 with value: 0.4949440515359768.


***Trial 8 : TrainTestSplit + lag_60 + rolling_avg_3 + rolling_std_3***

In [ ]:
# 2. Sort the dataframe by the full date (oldest to latest)
df8=df.copy()
df8 = df8.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df8['rolling_std'] = df8['daily_columno3'].shift(1).rolling(window=3).std()
df8['rolling_avg'] = df5['daily_columno3'].shift(1).rolling(window=3).mean()


for i in range(1, 61):
    df8[f'lag{i}'] = df8['daily_columno3'].shift(i)


df8.dropna(inplace=True)
print(df8.shape)
df8.head()

(65447, 64)


,daily_date,daily_columno3,rolling_std,rolling_avg,lag1,lag2,lag3,lag4,lag5,lag6,...,lag51,lag52,lag53,lag54,lag55,lag56,lag57,lag58,lag59,lag60
63,1980-03-06,323.0,59.911045,357.333333,424.0,308.0,340.0,286.0,287.4,297.2,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
64,1980-03-12,373.0,63.089883,351.666667,323.0,424.0,308.0,340.0,286.0,287.4,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7
65,1980-03-13,363.0,50.500825,373.333333,373.0,323.0,424.0,308.0,340.0,286.0,...,295.2,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8
66,1980-03-24,381.0,26.457513,353.000000,363.0,373.0,323.0,424.0,308.0,340.0,...,305.9,295.2,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8
67,1980-03-25,333.0,9.018500,372.333333,381.0,363.0,373.0,323.0,424.0,308.0,...,307.9,305.9,295.2,296.2,342.0,365.4,332.2,332.2,309.8,330.3


In [ ]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 61)]
lag_features.append('rolling_std')
lag_features.append('rolling_avg')
X = df8[lag_features]
y = df8['daily_columno3']


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
X_train = minmax_x.fit_transform(X_train)
y_train = minmax_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
X_test = minmax_x.transform(X_test)
y_test = minmax_y.transform(y_test.values.reshape(-1, 1)).flatten()

# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results[name] = {
        'Model': model,
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002578 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15810
[LightGBM] [Info] Number of data points in the train set: 52357, number of used features: 62
[LightGBM] [Info] Start training from score 0.000000
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.647
MAE : 0.470
R2  : 52.92 %

----- Decision Tree -----
RMSE: 0.927
MAE : 0.670
R2  : 3.27 %

----- Random Forest -----
RMSE: 0.650
MAE : 0.474
R2  : 52.44 %

----- Gradient Boosting -----
RMSE: 0.650
MAE : 0.471
R2  : 52.47 %

----- Support Vector Regressor -----
RMSE: 0.669
MAE : 0.482
R2  : 49.57 %

----- XGBoost -----
RMSE: 0.672
MAE : 0.481
R2  : 49.26 %

----- LightGBM -----
RMSE: 0.649
MAE : 0.471
R2  : 52.64 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


**Tuning 8**

In [ ]:

def tune_model(name, model_class, param_space):
    def objective(trial):
        params = {key: suggest(trial, key, dist) for key, dist in param_space.items()}
        model = model_class(**params)

        scores = []
        for train_idx, val_idx in cv.split(X_train):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            rmse = mean_squared_error(y_val, preds)
            scores.append(rmse)
        return np.mean(scores)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30, show_progress_bar=True)
    print(f"{name} Best Params: {study.best_params}")
    return model_class(**study.best_params)

def suggest(trial, name, dist):
    if isinstance(dist, tuple) and dist[0] == "int":
        return trial.suggest_int(name, *dist[1:])
    elif isinstance(dist, tuple) and dist[0] == "float":
        return trial.suggest_float(name, *dist[1:])
    elif isinstance(dist, tuple) and dist[0] == "cat":
        return trial.suggest_categorical(name, dist[1])
    else:
        raise ValueError(f"Unknown dist type for {name}: {dist}")

# Define parameter search spaces
param_spaces = {
    "XGBoost": {
        "tree_method": ("cat", ["gpu_hist"]),
        "predictor": ("cat", ["gpu_predictor"]),
        "n_estimators": ("int", 100, 150),
        "max_depth": ("int", 3, 7),
        "learning_rate": ("float", 0.01, 0.05),
        "subsample": ("float", 0.6, 0.9),
        "colsample_bytree": ("float", 0.6, 0.9)
    },
    "LightGBM": {  # Remove GPU
        "n_estimators": ("int", 100, 150),
        "learning_rate": ("float", 0.01, 0.05),
        "max_depth": ("int", 3, 7),
        "num_leaves": ("int", 15, 31),
        "subsample": ("float", 0.6, 0.9)
    },
    "Gradient Boosting": {
        "n_estimators": ("int", 100, 150),
        "learning_rate": ("float", 0.01, 0.1),
        "max_depth": ("int", 3, 5)
    },
    'Decision Tree': {
        'max_depth': ('int', 3, 10),
        'min_samples_split': ('int', 2, 10),
        'min_samples_leaf': ('int', 1, 5)
    },
    'Random Forest': {
        'n_estimators': ('int', 100, 200),
        'max_depth': ('int', 3, 10),
        'min_samples_split': ('int', 2, 10),
        'min_samples_leaf': ('int', 1, 5)
    },
    'Support Vector Regressor': {
        'kernel': ('cat', ['rbf']),
        'C': ('float', 0.1, 10.0),
        'epsilon': ('float', 0.01, 0.1)
    }
}

model_classes = {
    "XGBoost": XGBRegressor,
    "LightGBM": LGBMRegressor,
    "Gradient Boosting": GradientBoostingRegressor,
    'Decision Tree': DecisionTreeRegressor,
    'Random Forest': RandomForestRegressor,
    'Support Vector Regressor': SVR,
}

best_models = {}
results = {}

for name in model_classes:
    print(f"\nTuning {name}...")
    model = tune_model(name, model_classes[name], param_spaces[name])
    model.fit(X_train, y_train)
    best_models[name] = model

    # Evaluation
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# Print results
print("\nModel Evaluation Metrics:\n")
for name, metrics in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {metrics['RMSE']:.3f}")
    print(f"MAE : {metrics['MAE']:.3f}")
    print(f"R2  : {metrics['R2']*100:.2f} %\n")